In [4]:
!pip install tensorflow pandas numpy scikit-learn

  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-py2.py3-none-win_amd64.whl.metadata (5.3 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached namex-0.1.0-py3-none-any.whl.metadata (322 bytes)
   ---------------------------------------- 0.0/351.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/351.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/351.2 MB ? eta -:--:--
   ---------------------------------------- 0.3/351.2 MB ? eta -:--:--
   ---------------------------------------- 0.5/351.2 MB 749.3 kB/s eta 0:07:48
   ---------------------------------------- 0.5/351.2 MB 749.3 kB/s eta 0:07:48
   ---------------------------------------- 0.5/351.2 MB 749.3 kB/s eta 0:07:48
   ---------------------------------------- 0.8/351.2 MB 657.8 kB/s eta 0:08:53
   ---------------------------------------- 0

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.45.1 requires protobuf<7,>=3.20, but you have protobuf 7.35.1 which is incompatible.


In [17]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.utils import class_weight   
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

 
df = pd.read_csv('spamdata.csv', encoding='latin-1')
df.dropna(subset=['Category', 'Message'], inplace=True)
df['label'] = df['Category'].map({'spam': 1, 'ham': 0})

X = df['Message'].values
y = df['label'].values

X_train_text, X_test_text, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

max_words = 5000  
max_len = 100     

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train_text)

X_train_padded = pad_sequences(tokenizer.texts_to_sequences(X_train_text), maxlen=max_len, padding='pre')
X_test_padded = pad_sequences(tokenizer.texts_to_sequences(X_test_text), maxlen=max_len, padding='pre')

 
from tensorflow.keras.layers import GlobalMaxPooling1D

model = Sequential([
    Embedding(input_dim=max_words, output_dim=64),
    LSTM(64, return_sequences=True), 
    GlobalMaxPooling1D(),             
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
print("\nTraining Robust LSTM Model...")
model.fit(
    X_train_padded, y_train, 
    epochs=6,  
    batch_size=64, 
    validation_data=(X_test_padded, y_test)
)

loss, accuracy = model.evaluate(X_test_padded, y_test)
print(f"\nFixed LSTM Test Accuracy: {accuracy * 100:.2f}%")

 
def predict_spam(custom_message):
    seq = tokenizer.texts_to_sequences([custom_message])
    padded = pad_sequences(seq, maxlen=max_len, padding='pre')
    prediction = model.predict(padded, verbose=0)[0][0]
    print(f"Spam Probability: {prediction * 100:.2f}%") 
    return "SPAM 🚨" if prediction > 0.5 else "HAM (Legitimate) ✅"

print("\n--- Testing Custom Messages ---")
print(f"Prediction: {predict_spam('URGENT! You have won a 1-week all-expenses-paid trip to Hawaii. Text WON to 88712.')}\n")
print(f"Prediction: {predict_spam('Hey, are you free to jump on a quick call to review the project slides?')}\n")

 


Training Robust LSTM Model...
Epoch 1/6
70/70 ━━━━━━━━━━━━━━━━━━━━ 5s 48ms/step - accuracy: 0.8620 - loss: 0.4271 - val_accuracy: 0.8664 - val_loss: 0.3490
Epoch 2/6
70/70 ━━━━━━━━━━━━━━━━━━━━ 3s 44ms/step - accuracy: 0.9295 - loss: 0.2081 - val_accuracy: 0.9865 - val_loss: 0.0731
Epoch 3/6
70/70 ━━━━━━━━━━━━━━━━━━━━ 3s 42ms/step - accuracy: 0.9850 - loss: 0.0611 - val_accuracy: 0.9883 - val_loss: 0.0485
Epoch 4/6
70/70 ━━━━━━━━━━━━━━━━━━━━ 3s 42ms/step - accuracy: 0.9926 - loss: 0.0348 - val_accuracy: 0.9865 - val_loss: 0.0646
Epoch 5/6
70/70 ━━━━━━━━━━━━━━━━━━━━ 3s 36ms/step - accuracy: 0.9953 - loss: 0.0245 - val_accuracy: 0.9892 - val_loss: 0.0441
Epoch 6/6
70/70 ━━━━━━━━━━━━━━━━━━━━ 3s 40ms/step - accuracy: 0.9964 - loss: 0.0175 - val_accuracy: 0.9910 - val_loss: 0.0482
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9910 - loss: 0.0482

Fixed LSTM Test Accuracy: 99.10%

--- Testing Custom Messages ---
Spam Probability: 99.74%
Prediction: SPAM 🚨

Spam Probability: 28.43%
Pr